In [ ]:
from typing import Optional
import argparse, codecs, json, logging, os, pandas as pd, requests, sys, time

In [ ]:
excel_file_path = r"C:\Users\youne\Desktop\Biyectiva\Proyectos\cloudskin\app\storage\app\public\datasets\d1dc407d-7d47-492a-b774-d13668e88f0b\dataFile\cwb8ZqSRXLNH2ml97wgDIDkoEUmoaMBuuldQNVQe.xlsx"
url=r"http://localhost:8080/api/datasets/d1dc407d-7d47-492a-b774-d13668e88f0b"
latitude=0
longitude=0

# Asumiendo que ya tienes el 'excel_file_path' definido y cargado en 'df'
df = pd.read_excel(excel_file_path)

print("Antes de la conversión:")
print(df.head())

data1 = df.to_dict(orient='records')

print(data1)

for columna in df.columns:
    if pd.api.types.is_datetime64_ns_dtype(df[columna]):
        df[columna] = df[columna].astype(str)

print("\nDespués de la conversión:")
print(df.head())

data = df.to_dict(orient='records')

print(data)

In [ ]:
def replace_escape_chars(key):
    return codecs.decode(key.encode('latin-1'), 'unicode-escape')

grouped_data = []
for record in data:
    processed_record = {}
    temp_key = None
    temp_record = {}

    for key, value in record.items():
        if key.startswith('TER') or key.startswith('MPS'):
            if temp_key is not None:
                processed_record[temp_key] = temp_record
            temp_key = key
            temp_record = {key: value}
        elif temp_key is not None:
            temp_record[key] = value
        else:
            processed_record[key] = value

    processed_record[temp_key] = temp_record
    grouped_data.append(processed_record)

first_record = grouped_data[0]
results = {}

print(first_record)
print(grouped_data)

In [ ]:
keys = [clave for clave in first_record.keys() if clave not in ["Unnamed: 0", "Unnamed: 1", None]]
keys

In [ ]:
first_record

In [ ]:
Key = keys[0]
print(Key)
row = grouped_data[1]
print(row)
data_keys = [replace_escape_chars(valor) for valor in first_record.keys()] + [replace_escape_chars(first_record["Unnamed: 0"]), replace_escape_chars(first_record["Unnamed: 1"])]
print(data_keys)

data_values = [replace_escape_chars(value) for value in row.values()] + [row["Unnamed: 0"], row["Unnamed: 1"]]
print(data_values)
data = {k: v for k, v in zip(data_keys, data_values)}

print(data)

In [ ]:
for record in grouped_data:
    start_time = time.time()
    keys = [key for key in record.keys() if key not in ["Unnamed: 0", "Unnamed: 1", None]]
    for key in keys:
        if key not in results:
            results[key] = []
        
        data_value = record[key]

        result = {
            "latitude": latitude,
            "longitude": longitude,
            "data": data_value
        }

        results[key].append(result)

#         if url:
#             response = requests.post(url, json=result)
#             if response.status_code == 200:
#                 logging.info("Data successfully uploaded.")
#             else:
#                 logging.error(f"Failed to upload data: {response.status_code}, {response.text}")
#             elapsed_time = time.time() - start_time
#             logging.info(f"Individual request execution time: {elapsed_time} seconds")
#     elapsed_time = time.time() - start_time
#     logging.info(f"Total block execution time: {elapsed_time} seconds")
# logging.info(f"Finished processing file: {excel_file_path}")

print(results)

In [ ]:
print(result)

In [ ]:
print("Procesamiento del archivo Excel iniciado...")
print(url, "\n", result)
response = requests.post(url, json=result)
if response.status_code == 200:
    print("Data successfully uploaded.")
else:
    print(f"Failed to upload data: {response.status_code}, {response.text}")

In [ ]:
import requests
import pandas as pd

class DataUploader:
    def __init__(self, excel_file_path: str, url: str, dataset_id: str):
        self.excel_file_path = excel_file_path
        self.url = url
        self.dataset_id = dataset_id

    def read_excel_file(self) -> pd.DataFrame:
        return pd.read_excel(self.excel_file_path)

    def prepare_and_send_data(self) -> None:
        df = self.read_excel_file()
        data_list = df.to_dict(orient='records')

        for data in data_list:
            self.send_data(data)

    def send_data(self, data: dict) -> None:
        # Asumimos que la latitud y longitud están incluidas en los datos, ajusta según sea necesario.
        payload = {
            "data": data,
            "latitude": data.get("latitude"),  # Ajusta la clave según tus datos
            "longitude": data.get("longitude")  # Ajusta la clave según tus datos
        }

        response = requests.post(f"{self.url}/{self.dataset_id}", json=payload)

        if response.status_code == 200:
            print("Data successfully uploaded.")
        else:
            print(f"Failed to upload data: {response.status_code}, {response.text}")

# Ejemplo de uso
uploader = DataUploader("path_to_your_excel.xlsx", "http://yourserver.com/api/addDataRead", "your_dataset_id")
uploader.prepare_and_send_data()

In [2]:
import json
import requests
import logging
import pandas as pd
from typing import Optional

class ExcelDataUploader:
    def __init__(self, excel_file_path: str, url: str, latitude: Optional[float] = None, longitude: Optional[float] = None):
        self.excel_file_path = excel_file_path
        self.url = url
        self.latitude = latitude
        self.longitude = longitude

    def process_excel(self):
        df = pd.read_excel(self.excel_file_path)
        for columna in df.columns:
            if pd.api.types.is_datetime64_ns_dtype(df[columna]):
                df[columna] = df[columna].astype(str)

        data_records = df.to_dict(orient='records')
        self._send_data(data_records)

    def _send_data(self, data_records):
        try:
            payload = {
                "data": data_records,
                "latitude": self.latitude,
                "longitude": self.longitude
            }
            print(payload)
            response = requests.post(self.url, json=payload)
            response.raise_for_status()
            logging.info("Data sent successfully")
        except requests.exceptions.RequestException as e:
            logging.error(f"Error sending data: {e}")

def main():
    excel_file_path = r"C:\Users\youne\Desktop\Biyectiva\Proyectos\cloudskin\app\storage\app\public\datasets\d1dc407d-7d47-492a-b774-d13668e88f0b\dataFile\cwb8ZqSRXLNH2ml97wgDIDkoEUmoaMBuuldQNVQe.xlsx"
    url=r"http://localhost:8080/api/datasets/d1dc407d-7d47-492a-b774-d13668e88f0b"
    latitude=39.340795
    longitude=-1.925014

    uploader = ExcelDataUploader(excel_file_path, url, latitude, longitude)
    uploader.process_excel()

if __name__ == "__main__":
    main()

{'data': [{'Fecha': '2024-02-15', 'Temperatura': 10.7, 'Humedad': 45.6}, {'Fecha': '2024-02-16', 'Temperatura': 13.15, 'Humedad': 17.6}, {'Fecha': '2024-02-17', 'Temperatura': 12.61, 'Humedad': 55.7}, {'Fecha': '2024-02-18', 'Temperatura': 12.25, 'Humedad': 45.6}, {'Fecha': '2024-02-19', 'Temperatura': 14.7, 'Humedad': 45.6}, {'Fecha': '2024-02-20', 'Temperatura': 8.15, 'Humedad': 17.6}, {'Fecha': '2024-02-21', 'Temperatura': 5.61, 'Humedad': 55.7}, {'Fecha': '2024-02-22', 'Temperatura': 11.25, 'Humedad': 45.6}], 'latitude': 39.340795, 'longitude': -1.925014}
